In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from PPairS.constants import collated_results_path as clp
from PPairS.constants import graph_path
from PPairS.utils import families

In [2]:
datasets = ["newsroom", "summeval", "hanna"]
datatitles = ["NEWSROOM", "SummEval", "HANNA"]
for dataset, title in zip(datasets, datatitles):
    probe_u = pd.read_json(f'{clp}/{dataset}/probe_u_results.jsonl', orient='records', lines=True).set_index('model')
    prompting = pd.read_json(f'{clp}/{dataset}/results.jsonl', orient='records', lines=True).set_index('model')

    # Create the plot
    fig, ax = plt.subplots(figsize=(13, 5))

    # Track x positions for each family
    x_pos = 0
    xticks = []
    xlabels = []
    minor_xticks = []
    minor_xticklabels = []

    # Colors for consistency
    colors = {
        'pairwise': '#1f77b4',
        'unsup': '#ff7f0e',
        'geval': '#2ca02c',
        'direct': '#d62728'
    }

    family_full = {
        'mistral': 'Mistral',
        'llama': 'Llama 3.1',
        'qwen': 'Qwen 2.5',
        'gemma': 'Gemma 2'
    }

    for family, models in families.items():
        # Get data points for this family
        x_coords = range(x_pos, x_pos + len(models))
        
        # Get scores for all methods
        pc_scores = prompting[prompting['method'] == 'pairwise_comparisons'].loc[models, 'avg_f1'].values
        unsup_scores = probe_u.loc[models, 'avg_f1'].values
        geval_scores = prompting[prompting['method'] == 'g_eval'].loc[models, 'avg_f1'].values
        direct_scores = prompting[prompting['method'] == 'direct_scoring'].loc[models, 'avg_f1'].values
        
        # Plot lines
        ax.plot(x_coords, pc_scores, marker='o', label='pairwise-comparisons' if x_pos == 0 else "", color=colors['pairwise'])
        ax.plot(x_coords, unsup_scores, marker='s', label='u-probe' if x_pos == 0 else "", color=colors['unsup'])
        ax.plot(x_coords, geval_scores, marker='^', label='g-eval' if x_pos == 0 else "", color=colors['geval'])
        ax.plot(x_coords, direct_scores, marker='D', label='direct-scoring' if x_pos == 0 else "", color=colors['direct'])
        
        # Track positions for labels
        xticks.append(x_pos + len(models)/2 - 0.5)
        xlabels.append(family_full[family])
        
        # Add minor ticks for model sizes
        for i, model in enumerate(models):
            minor_xticks.append(x_pos + i)
            size = model.split('-')[-1]
            minor_xticklabels.append(size)
            
        # Update x position for next family
        x_pos += len(models) + 2  # Add gap between families

    # Customize the plot
    ax.xaxis.remove_overlapping_locs = False
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=16, y=-0.075)
    ax.set_xticks(minor_xticks, minor=True)
    ax.set_xticklabels(minor_xticklabels, minor=True, fontsize=14)
    ax.set_ylabel('F1 Score', fontsize=16)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=16)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_title(f'Unsupervised Probes: {title}', fontsize=16)

    plt.tight_layout()
    # plt.show()
    plt.savefig(f'{graph_path}/unsupervised_probes_{dataset}.png', dpi=400)
    plt.close()

In [3]:
# Read and average data across text quality datasets
datasets = ['newsroom', 'summeval', 'hanna']
dataset_averages_probe_u = []
dataset_averages_prompting = []

# First get averages for each dataset
for dataset in datasets:
    probe_u = pd.read_json(f'{clp}/{dataset}/probe_u_results.jsonl', orient='records', lines=True).set_index('model')
    prompting = pd.read_json(f'{clp}/{dataset}/results.jsonl', orient='records', lines=True).set_index('model')
    
    # Store the averages for this dataset
    dataset_averages_probe_u.append(probe_u)
    dataset_averages_prompting.append(prompting)

# Now average across datasets (average of averages)
probe_u_by_dataset = [df.groupby('model').mean() for df in dataset_averages_probe_u]
probe_u = sum(probe_u_by_dataset) / len(probe_u_by_dataset)

prompting_by_dataset = [df.groupby(['model', 'method']).mean() for df in dataset_averages_prompting]
prompting = sum(prompting_by_dataset) / len(prompting_by_dataset)
prompting = prompting.reset_index().set_index('model')

# Create the plot
fig, ax = plt.subplots(figsize=(13, 5))

# Track x positions for each family
x_pos = 0
xticks = []
xlabels = []
minor_xticks = []
minor_xticklabels = []

# Colors for consistency
colors = {
    'pairwise': '#1f77b4',
    'unsup': '#ff7f0e',
    'geval': '#2ca02c',
    'direct': '#d62728'
}

family_full = {
    'mistral': 'Mistral',
    'llama': 'Llama 3.1',
    'qwen': 'Qwen 2.5',
    'gemma': 'Gemma 2'
}

for family, models in families.items():
    # Get data points for this family
    x_coords = range(x_pos, x_pos + len(models))
    
    # Get scores for all methods
    pc_scores = prompting[prompting['method'] == 'pairwise_comparisons'].loc[models, 'avg_f1'].values
    unsup_scores = probe_u.loc[models, 'avg_f1'].values
    geval_scores = prompting[prompting['method'] == 'g_eval'].loc[models, 'avg_f1'].values
    direct_scores = prompting[prompting['method'] == 'direct_scoring'].loc[models, 'avg_f1'].values
    
    # Plot lines
    ax.plot(x_coords, pc_scores, marker='o', label='pairwise-comparisons' if x_pos == 0 else "", color=colors['pairwise'])
    ax.plot(x_coords, unsup_scores, marker='s', label='u-probe' if x_pos == 0 else "", color=colors['unsup'])
    ax.plot(x_coords, geval_scores, marker='^', label='g-eval' if x_pos == 0 else "", color=colors['geval'])
    ax.plot(x_coords, direct_scores, marker='D', label='direct-scoring' if x_pos == 0 else "", color=colors['direct'])
    
    # Track positions for labels
    xticks.append(x_pos + len(models)/2 - 0.5)
    xlabels.append(family_full[family])
    
    # Add minor ticks for model sizes
    for i, model in enumerate(models):
        minor_xticks.append(x_pos + i)
        size = model.split('-')[-1]
        minor_xticklabels.append(size)
    
    # Update x position for next family
    x_pos += len(models) + 2  # Add gap between families

# Customize the plot
ax.xaxis.remove_overlapping_locs = False
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels, fontsize=16, y=-0.075)
ax.set_xticks(minor_xticks, minor=True)
ax.set_xticklabels(minor_xticklabels, minor=True, fontsize=14)
ax.set_ylabel('F1 Score', fontsize=16)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=16)
ax.tick_params(axis='y', labelsize=14)
ax.set_title('Unsupervised Probes: Text Quality', fontsize=16)

plt.tight_layout()
# plt.show()
plt.savefig(f'{graph_path}/unsupervised_probes_text_quality.png', dpi=400)
plt.close()